# Geocoding the healthcare facilities

This notebook geocodes the healthcare facilities identified in the outpatient records to obtain their geographic coordinates and address information.

The resulting dataset is used to map healthcare visits within the city of São Paulo.

# Input

- Dataset containing the healthcare facilities extracted from the SIA/CNES records

# Outputs

This notebook enriches the dataset with geographic information, including:

- Latitude and longitude
- Formatted address
- Street name
- Neighborhood
- ZIP code returned by the geocoding service

In [1]:
import pandas as pd

In [2]:
df_capital_city = pd.read_csv("df_capital_city.csv")

In [3]:
df_capital_city.columns

Index(['ano', 'mes', 'sigla_uf', 'nome_uf', 'id_municipio', 'nome_municipio',
       'id_estabelecimento_cnes', 'cid_principal_subcategoria',
       'cid_secundario_subcategoria', 'cid_causas_associadas_subcategoria',
       'sexo_paciente', 'idade_paciente', 'raca_cor_paciente',
       'indicador_uf_residencia_paciente',
       'indicador_municipio_residencia_paciente', 'id_municipio_cnes',
       'nome_municipio_cnes', 'cep', 'indicador_vinculo_sus', 'faixa_etaria'],
      dtype='object')

In [5]:
df_capital_city["nome_municipio_cnes"].unique()

array(['São Paulo'], dtype=object)

In [6]:
df_capital_city["cep"].dtype

dtype('int64')

In [7]:
df_capital_city["cep"].isna().sum()

0

## Prepare ZIP codes for geocoding

Convert the ZIP code ("CEP") field to a standardized eight-digit format and create a list of unique ZIP codes.

Geocoding only unique ZIP codes reduces the number of requests sent to the geocoding services and improves the overall processing time.

In [8]:
df_capital_city["cep"] = df_capital_city["cep"].astype(str)

In [9]:
df_capital_city["cep"].str.len()

0       7
1       7
2       7
3       7
4       7
       ..
2336    7
2337    7
2338    7
2339    7
2340    7
Name: cep, Length: 2341, dtype: int64

In [10]:
df_capital_city["cep"] = (
    df_capital_city["cep"]
    .astype(str)
    .str.zfill(8)
)

In [11]:
unique_postcodes = (
    df_capital_city[["cep"]]
    .drop_duplicates()
    .copy()
)

In [12]:
unique_postcodes

,cep
0,05401000
1,03427070
9,04463030
11,03163040
12,03976020
19,01332000
21,04231030
26,03227000
27,04217000
28,04757120


In [13]:
print("Records:", len(df_capital_city))
print("Unique postcodes:", len(unique_postcodes))

Records: 2341
Unique postcodes: 38


In [14]:
unique_postcodes["cep"].duplicated().sum()

0

## Geocode healthcare facilities

Use the ZIP codes associated with each healthcare facility to retrieve geographic coordinates and address information.

The geocoding process combines the ViaCEP API, which provides the official address for each ZIP code, with the Nominatim geocoding service, which converts the address into geographic coordinates.

In [15]:
pip install geopy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json

import requests
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim

In [17]:
geolocator = Nominatim(user_agent="sp_health_facilities_project")

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1,
    swallow_exceptions=True
)

## Retrieve addresses from ViaCEP

Use the zip code associated with each healthcare facility to retrieve address information from the ViaCEP API.

In [18]:
def geocode_postcode(postcode):
    viacep_url = f"https://viacep.com.br/ws/{postcode}/json/"

    try:
        response = requests.get(viacep_url, timeout=10)
        response.raise_for_status()
        postcode_data = response.json()

    except (requests.RequestException, ValueError) as error:
        print(f"Could not retrieve postcode {postcode}: {error}")

        return pd.Series({
            "latitude": None,
            "longitude": None,
            "formatted_address": None,
            "street": None,
            "neighborhood": None,
            "returned_postcode": None
        })

    if postcode_data.get("erro"):
        print(f"Invalid postcode: {postcode}")

        return pd.Series({
            "latitude": None,
            "longitude": None,
            "formatted_address": None,
            "street": None,
            "neighborhood": None,
            "returned_postcode": None
        })

    street = postcode_data.get("logradouro")
    neighborhood = postcode_data.get("bairro")
    returned_postcode = postcode_data.get("cep")

    search_address = ", ".join(
        part
        for part in [
            street,
            neighborhood,
            returned_postcode,
            "São Paulo",
            "SP",
            "Brasil"
        ]
        if part
    )

    location = geocode(
        search_address,
        country_codes="br",
        addressdetails=True,
        exactly_one=True
    )

    if location is not None:
        address = location.raw.get("address", {})

        returned_city = (
            address.get("city")
            or address.get("municipality")
            or address.get("town")
            or address.get("village")
        )

        if returned_city != "São Paulo":
            location = None

    if location is None:
        fallback_address = ", ".join(
            part
            for part in [
                returned_postcode,
                "São Paulo",
                "SP",
                "Brasil"
            ]
            if part
        )

        location = geocode(
            fallback_address,
            country_codes="br",
            addressdetails=True,
            exactly_one=True
        )

    if location is None:
        print(
            f"Could not geocode postcode {postcode} "
            f"({street}, {neighborhood}, São Paulo-SP)"
        )

        return pd.Series({
            "latitude": None,
            "longitude": None,
            "formatted_address": None,
            "street": street,
            "neighborhood": neighborhood,
            "returned_postcode": returned_postcode
        })

    return pd.Series({
        "latitude": location.latitude,
        "longitude": location.longitude,
        "formatted_address": location.address,
        "street": street,
        "neighborhood": neighborhood,
        "returned_postcode": returned_postcode
    })

In [21]:
from tqdm.auto import tqdm

tqdm.pandas(desc="Geocoding postcodes")

geocoded = unique_postcodes["cep"].progress_apply(geocode_postcode)

unique_postcodes[
    [
        "latitude",
        "longitude",
        "formatted_address",
        "street",
        "neighborhood",
        "returned_postcode"
    ]
] = geocoded

Geocoding postcodes:   0%|          | 0/38 [00:00<?, ?it/s]

In [38]:
print(len(unique_postcodes))

38


## Manually correct unresolved locations

A small number of ZIP codes could not be accurately geocoded automatically. For these cases, the geographic coordinates were manually verified and assigned to ensure that the corresponding healthcare facilities were correctly represented in the city map.

In [23]:
manual_coordinates = {
    "04463030": (-23.694302, -46.659430),
    "04313160": (-23.637816, -46.639402),
    "04316060": (-23.640625, -46.634090),
    "04852040": (-23.760233, -46.660991),
}

for postcode, (lat, lon) in manual_coordinates.items():
    unique_postcodes.loc[
        unique_postcodes["cep"] == postcode,
        ["latitude", "longitude"]
    ] = [lat, lon]

In [24]:
df_capital_city = df_capital_city.merge(
    unique_postcodes,
    on="cep",
    how="left"
)

In [25]:
df_capital_city.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['ano', 'mes', 'sigla_uf', 'nome_uf', 'id_municipio', 'nome_municipio',
       'id_estabelecimento_cnes', 'cid_principal_subcategoria',
       'cid_secundario_subcategoria', 'cid_causas_associadas_subcategoria',
       'sexo_paciente', 'idade_paciente', 'raca_cor_paciente',
       'indicador_uf_residencia_paciente',
       'indicador_municipio_residencia_paciente', 'id_municipio_cnes',
       'nome_municipio_cnes', 'cep', 'indicador_vinculo_sus', 'faixa_etaria',
       'latitude', 'longitude', 'formatted_address', 'street', 'neighborhood',
       'returned_postcode'],
      dtype='object')>

In [26]:
df_capital_city[
    df_capital_city["latitude"].isna() |
    df_capital_city["longitude"].isna()
]

,ano,mes,sigla_uf,nome_uf,id_municipio,nome_municipio,id_estabelecimento_cnes,cid_principal_subcategoria,cid_secundario_subcategoria,cid_causas_associadas_subcategoria,...,nome_municipio_cnes,cep,indicador_vinculo_sus,faixa_etaria,latitude,longitude,formatted_address,street,neighborhood,returned_postcode


## Assign healthcare facilities to districts

Convert the geocoded healthcare facilities into spatial points and compare their coordinates with the São Paulo district boundaries from the official district GeoJSON.

A spatial join is used to identify the district containing each healthcare facility, allowing the outpatient records to be analyzed and mapped by district.

In [27]:
import geopandas as gpd

In [28]:
districts = gpd.read_file("BAIRROS_SAOPAULO.geojson")

In [29]:
# Checking the coordinates
districts.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [30]:
gdf_capital_city = gpd.GeoDataFrame(
    df_capital_city.copy(),
    geometry=gpd.points_from_xy(
        df_capital_city["longitude"],
        df_capital_city["latitude"]
    ),
    crs="EPSG:4326"
)

In [31]:
gdf_capital_city = gdf_capital_city.to_crs(districts.crs)

In [32]:
gdf_capital_city = gpd.sjoin(
    gdf_capital_city,
    districts[["ds_nome", "geometry"]],
    how="left",
    predicate="within"
)

In [39]:
gdf_capital_city[
    ["formatted_address", "neighborhood", "ds_nome"]
].sample(10)

,formatted_address,neighborhood,ds_nome
2192,"Rua Doutor Ovídio Pires de Campos, Cerqueira C...",Cerqueira César,JARDIM PAULISTA
719,"Rua Doutor Ovídio Pires de Campos, Cerqueira C...",Cerqueira César,JARDIM PAULISTA
704,"Rua Doutor Ovídio Pires de Campos, Cerqueira C...",Cerqueira César,JARDIM PAULISTA
2170,"Rua Doutor Ovídio Pires de Campos, Cerqueira C...",Cerqueira César,JARDIM PAULISTA
1883,"Rua Doutor Ovídio Pires de Campos, Cerqueira C...",Cerqueira César,JARDIM PAULISTA
1723,"Rua Doutor Ovídio Pires de Campos, Cerqueira C...",Cerqueira César,JARDIM PAULISTA
1275,"Rua Doutor Ovídio Pires de Campos, Cerqueira C...",Cerqueira César,JARDIM PAULISTA
2073,"Rua Doutor Ovídio Pires de Campos, Cerqueira C...",Cerqueira César,JARDIM PAULISTA
1829,"Rua Doutor Ovídio Pires de Campos, Cerqueira C...",Cerqueira César,JARDIM PAULISTA
540,"Rua Doutor Ovídio Pires de Campos, Cerqueira C...",Cerqueira César,JARDIM PAULISTA


In [34]:
gdf_capital_city[
    gdf_capital_city["ds_nome"].isna()
]

,ano,mes,sigla_uf,nome_uf,id_municipio,nome_municipio,id_estabelecimento_cnes,cid_principal_subcategoria,cid_secundario_subcategoria,cid_causas_associadas_subcategoria,...,faixa_etaria,latitude,longitude,formatted_address,street,neighborhood,returned_postcode,geometry,index_right,ds_nome


In [35]:
gdf_capital_city.loc[
    gdf_capital_city["ds_nome"].isna(),
    [
        "nome_municipio_cnes",
        "id_estabelecimento_cnes",
        "cep",
        "street",
        "neighborhood",
        "formatted_address",
        "latitude",
        "longitude"
    ]
].drop_duplicates()

,nome_municipio_cnes,id_estabelecimento_cnes,cep,street,neighborhood,formatted_address,latitude,longitude


In [36]:
gdf_capital_city.loc[
    gdf_capital_city["ds_nome"].isna(),
    "cep"
].value_counts()

Series([], Name: count, dtype: int64)

In [41]:
missing_postcodes = (
    gdf_capital_city.loc[
        gdf_capital_city["ds_nome"].isna(),
        "cep"
    ]
    .unique()
)

In [42]:
df_capital_city.loc[
    df_capital_city["cep"].isin(missing_postcodes),
    [
        "nome_municipio_cnes",
        "id_estabelecimento_cnes",
        "cep"
    ]
]

,nome_municipio_cnes,id_estabelecimento_cnes,cep


In [44]:
gdf_capital_city.loc[
    gdf_capital_city["ds_nome"].isna(),
    [
        "cep",
        "latitude",
        "longitude"
    ]
].drop_duplicates()

,cep,latitude,longitude


In [53]:
gdf_capital_city["ds_nome"].isna().sum()

0

In [61]:
len(gdf_capital_city)

2341

In [69]:
len(df_capital_city), len(gdf_capital_city)

(2341, 2341)

In [70]:
gdf_capital_city.loc[
    gdf_capital_city["returned_postcode"].isin([
        "04917-070",
        "02042-000"
    ]),
    [
        "cep",
        "returned_postcode",
        "formatted_address",
        "neighborhood",
        "ds_nome",
        "latitude",
        "longitude"
    ]
]

,cep,returned_postcode,formatted_address,neighborhood,ds_nome,latitude,longitude
37,02042000,02042-000,"02042-000, Santana, São Paulo, Região Sudeste,...",Jardim São Paulo(Zona Norte),SANTANA,-23.493841,-46.616712
2313,04917070,04917-070,"Brasil, 414, Rua Santa Ifigênia, Santa Ifigêni...",Jardim Souza,JARDIM SAO LUIS,-23.538321,-46.639162
2314,04917070,04917-070,"Brasil, 414, Rua Santa Ifigênia, Santa Ifigêni...",Jardim Souza,JARDIM SAO LUIS,-23.538321,-46.639162


In [71]:
df_capital_city = pd.DataFrame(
    gdf_capital_city.drop(columns=["geometry", "index_right"])
)

In [72]:
df_capital_city

,ano,mes,sigla_uf,nome_uf,id_municipio,nome_municipio,id_estabelecimento_cnes,cid_principal_subcategoria,cid_secundario_subcategoria,cid_causas_associadas_subcategoria,...,cep,indicador_vinculo_sus,faixa_etaria,latitude,longitude,formatted_address,street,neighborhood,returned_postcode,ds_nome
0,2015,8,SP,São Paulo,3550308,São Paulo,2078015,F630,NaN,NaN,...,05401000,1,30–44,-23.557181,-46.666671,"Avenida Rebouças, Cerqueira César, Jardim Paul...",Avenida Rebouças,Cerqueira César,05401-000,JARDIM PAULISTA
1,2015,8,SP,São Paulo,3550308,São Paulo,3384292,NaN,NaN,F630,...,03427070,1,45–59,-23.555513,-46.537857,"Rua Boicininga, Vila Fernandes, Vila Carrão, C...",Rua Boicininga,Vila Carrão,03427-070,CARRAO
2,2015,11,SP,São Paulo,3550308,São Paulo,3384292,NaN,NaN,F630,...,03427070,1,45–59,-23.555513,-46.537857,"Rua Boicininga, Vila Fernandes, Vila Carrão, C...",Rua Boicininga,Vila Carrão,03427-070,CARRAO
3,2015,11,SP,São Paulo,3550308,São Paulo,3384292,NaN,NaN,F630,...,03427070,1,45–59,-23.555513,-46.537857,"Rua Boicininga, Vila Fernandes, Vila Carrão, C...",Rua Boicininga,Vila Carrão,03427-070,CARRAO
4,2015,11,SP,São Paulo,3550308,São Paulo,3384292,NaN,NaN,F630,...,03427070,1,45–59,-23.555513,-46.537857,"Rua Boicininga, Vila Fernandes, Vila Carrão, C...",Rua Boicininga,Vila Carrão,03427-070,CARRAO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2336,2025,11,SP,São Paulo,3550308,São Paulo,5731194,F630,NaN,NaN,...,04463030,1,18–29,-23.694302,-46.659430,"Brasil, 414, Rua Santa Ifigênia, Santa Ifigêni...",Rua Conceição da Boa Viagem,Balneário Mar Paulista,04463-030,PEDREIRA
2337,2025,11,SP,São Paulo,3550308,São Paulo,5731194,F630,NaN,NaN,...,04463030,1,30–44,-23.694302,-46.659430,"Brasil, 414, Rua Santa Ifigênia, Santa Ifigêni...",Rua Conceição da Boa Viagem,Balneário Mar Paulista,04463-030,PEDREIRA
2338,2025,10,SP,São Paulo,3550308,São Paulo,6148387,F630,NaN,NaN,...,02634070,1,60+,-23.456940,-46.649928,"Rua Desembargador Rodrigues Sette, Vila Amélia...",Rua Desembargador Rodrigues Sette,Jardim Peri,02634-070,CACHOEIRINHA
2339,2025,4,SP,São Paulo,3550308,São Paulo,7861249,F630,NaN,NaN,...,02031200,1,0–17,-23.521010,-46.625197,"Avenida Cruzeiro do Sul, Canindé, Pari, São Pa...",Avenida Cruzeiro do Sul,Canindé,02031-200,PARI


In [73]:
df_capital_city.columns[df_capital_city.columns.duplicated()]

Index([], dtype='object')

## Aggregate outpatient records by district

Count the outpatient records associated with healthcare facilities in each São Paulo district.

The resulting dataset contains one row per district and the total number of outpatient records, making it suitable for joining with the district boundaries in the final choropleth map.

In [77]:
df_districts = (
    df_capital_city
    .dropna(subset=["ds_nome"])
    .groupby("ds_nome")
    .size()
    .reset_index(name="value")
    .sort_values("value", ascending=False)
)

In [80]:
df_districts

,ds_nome,value
9,JARDIM PAULISTA,2209
7,JABAQUARA,30
3,CARRAO,18
16,REPUBLICA,17
14,PEDREIRA,15
15,PIRITUBA,8
19,SAPOPEMBA,7
6,IPIRANGA,7
0,BELA VISTA,5
8,JARDIM ANGELA,4


In [81]:
df_districts.shape

(22, 2)

In [82]:
df_districts.to_csv(
    "df_capital_districts.csv",
    index=False
)